In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import platform
import h5py
import beautifuljason as bjason
import json

import numpy as np
import pandas as pd

In [14]:
fn_assigned = Path("alpha_ionone_assigned_by_jason.jjh5")
fn_unassigned = Path("alpha_ionone_unassigned.jjh5")

fn_simpleNMR_json = Path("alpha_ionone_assigned_by_jason_assignments_from_simplemnova.json")
                   

In [15]:
fn_simpleNMR_json.exists(), fn_simpleNMR_json.name

(True, 'alpha_ionone_assigned_by_jason_assignments_from_simplemnova.json')

In [16]:
with open(fn_simpleNMR_json, 'r') as f:
    json_data = json.load(f)

In [17]:
json_data.keys()

dict_keys(['nodes_orig', 'nodes_now', 'links', 'smilesString', 'molfile', 'dataFrom', 'oldjsondata', 'molgraph', 'shortest_paths', 'svg', 'catoms', 'catoms_orig', 'best_results', 'workingDirectory', 'workingFilename', 'title'])

In [18]:
json_data['nodes_now'][1]

{'atomNumber': 2,
 'id': 1,
 'iupacLabel': '',
 'jCouplingClass': '',
 'jCouplingVals': '',
 'numProtons': 1,
 'ppm': 148.92451256333828,
 'ppm_calculated': 148.25,
 'sym_atomNumber': '',
 'sym_atom_idx': '',
 'symbol': 'C',
 'visible': True,
 'x': 0.5,
 'y': 0.630666069386199,
 'H1_ppm': [6.592629998578599]}

In [19]:
nodes_now = {}
for node in json_data['nodes_now']:
    id = node['id']
    nodes_now[id] = node

In [24]:
with bjason.Document(fn_unassigned, mode="r+") as doc:
    mol = doc.mol_data[0]
    spec = mol.spectra[0]

    for shift in spec.shifts:

        print(shift.nums)
        shift.set_value_error_pair(
            bjason.Molecule.CalcMethod.Experimental,
            nodes_now[shift.nums[0]]['ppm'],
        )


[0]
[1]
[2]
[3]
[4]
[5]
[6]
[7]
[8]
[9]
[10]
[11]
[12]


In [43]:
fn_assigned.exists(), fn_unassigned.exists()

(True, True)

In [47]:
with bjason.Document(fn_assigned, mode="r") as doc:
    print(type(doc.mol_data), "\n", dir(doc.mol_data))

<class 'beautifuljason.data.Molecule.List'> 
 ['Iterator', '__bool__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_attr', '_len', 'append', 'h5_group', 'list_group_name', 'value_class']


In [ ]:

with bjason.Document(fn_unassigned, mode="r") as doc:
    print(f"number of molecules: {len(doc.mol_data)}")


    mol = doc.mol_data[0]
    print(f"number of spectra: {len(mol.spectra)}")
    # spec = mol.spectra[0]

    for ispec, spec in enumerate(mol.spectra):
        # print out the nuclues type of the spectra
        if spec.nucleus == bjason.Molecule.Atom.NuclType.H1:
            print(f"spectra {ispec} nucleus: H1")
        elif spec.nucleus == bjason.Molecule.Atom.NuclType.C13:
            print(f"spectra {ispec} nucleus: C13")
        else:
            continue

        for shift in spec.shifts:
            calc_shift_value, _ = shift.get_value_error_pair(
                bjason.Molecule.CalcMethod.Experimental, 
            )
            for num in shift.nums:
                element = "H" if spec.nucleus == bjason.Molecule.Atom.NuclType.H1 else "C"
                aid = element + str(num) + (f"({shift.mark})" if shift.mark else "")
                print(aid, calc_shift_value)

number of molecules: 1
number of spectra: 2
spectra 0 nucleus: C13
C0 197.2
C1 148.25
C2 132.45
C3 132.05
C4 122.55
C5 54.25
C6 32.4
C7 29.900002
C8 27.36111
C9 27.1
C10 27.36111
C11 23.044445
C12 22.862501
spectra 1 nucleus: H1
H8 0.86
H9 2.26
H10 0.86
H12 1.58
H1 6.5302734
H2 6.09
H4 5.46
H5 2.4887695
H7(dn) 1.2568359
H7(up) 1.5322266
H11(dn) 2.0493164
H11(up) 2.0493164


In [ ]:

with bjason.Document(fn_assigned, mode="r") as doc:
    mol = doc.mol_data[0]
    spec = mol.spectra[0]

    for shift in spec.shifts:
        value, _ = shift.get_value_error_pair(
            bjason.Molecule.CalcMethod., 
        )
        if value is not None:
            print(list(shift.nums), shift.nh, shift.mark, value, shift.value)

[np.uint32(0)] 0 None 197.2 [198.3108  198.01193 197.7539  197.2     197.2    ]
[np.uint32(1)] 1 None 148.25 [148.92451 147.93091 147.94922 148.25    148.25   ]
[np.uint32(2)] 1 None 132.45 [132.27106 133.10081 132.44629 132.45    132.45   ]
[np.uint32(3)] 0 None 132.05 [131.84015 132.65195 131.22559 132.05    132.05   ]
[np.uint32(4)] 1 None 122.55 [122.59451  122.61086  122.680664 122.55     122.55    ]
[np.uint32(5)] 1 None 54.25 [54.24827  56.50496  53.527832 54.25     54.25    ]
[np.uint32(6)] 0 None 32.4 [32.45796  34.327927 32.53174  32.4      32.4     ]
[np.uint32(7)] 2 None 29.900002 [31.172062 34.607113 30.151367 29.900002 29.900002]
[np.uint32(8)] 3 None 27.36111 [27.739805 26.21915  27.023315 27.36111  27.36111 ]
[np.uint32(9)] 3 None 27.1 [26.881624 27.89914  27.175903 27.1      27.1     ]
[np.uint32(10)] 3 None 27.36111 [26.746096 26.21915  27.023315 27.36111  27.36111 ]
[np.uint32(11)] 2 None 23.044445 [22.961699 22.827124 22.964478 23.044445 23.044445]
[np.uint32(12)] 3

In [15]:
dir(shift)

['List',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_attr',
 'acount',
 'error_spheres',
 'get_value_error_pair',
 'h5_group',
 'ignored_auto',
 'ignored_user',
 'is_exchangeable',
 'mark',
 'nh',
 'nums',
 'set_value_error_pair',
 'value',
 'value_error',
 'value_method',
 'value_spheres']

In [13]:
type(bjason.Molecule.CalcMethod.Experimental), dir(bjason.Molecule.CalcMethod.Experimental)

(<enum 'CalcMethod'>,
 ['Best',
  'Experimental',
  'Increment',
  'NeuralNetwork',
  'SphericCodes',
  'Undefined',
  '__abs__',
  '__add__',
  '__and__',
  '__bool__',
  '__ceil__',
  '__class__',
  '__copy__',
  '__deepcopy__',
  '__delattr__',
  '__dict__',
  '__dir__',
  '__divmod__',
  '__doc__',
  '__eq__',
  '__float__',
  '__floor__',
  '__floordiv__',
  '__format__',
  '__ge__',
  '__getattribute__',
  '__getnewargs__',
  '__getstate__',
  '__gt__',
  '__hash__',
  '__index__',
  '__init__',
  '__init_subclass__',
  '__int__',
  '__invert__',
  '__le__',
  '__lshift__',
  '__lt__',
  '__mod__',
  '__module__',
  '__mul__',
  '__ne__',
  '__neg__',
  '__new__',
  '__objclass__',
  '__or__',
  '__pos__',
  '__pow__',
  '__radd__',
  '__rand__',
  '__rdivmod__',
  '__reduce__',
  '__reduce_ex__',
  '__repr__',
  '__rfloordiv__',
  '__rlshift__',
  '__rmod__',
  '__rmul__',
  '__ror__',
  '__round__',
  '__rpow__',
  '__rrshift__',
  '__rshift__',
  '__rsub__',
  '__rtruediv__',


In [15]:
dir(doc)

['__class__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_create_item_elem',
 'all_logfile_paths',
 'close',
 'copy',
 'create_chart_item',
 'create_document_id',
 'create_image_data',
 'create_image_item',
 'create_mscentroid_table',
 'create_mspeaks_table',
 'create_nmrassignments_table',
 'create_nmrdosy_table',
 'create_nmrmultiplet_report',
 'create_nmrmultiplets_table',
 'create_nmrparams_table',
 'create_nmrpeaks_table',
 'create_params_table',
 'create_text_item',
 'custom_data',
 'document_id',
 'file_name',
 'h5_file',
 'image_data',
 'is_temporary',
 'items',
 'item

In [7]:
with bjason.Document(fn_unassigned, "r") as doc:
    mol = doc.mol_data[0]
    spec = mol.spectra[0]

    for shift in spec.shifts:
        value, _ = shift.get_value_error_pair(
            bjason.Molecule.CalcMethod.Experimental
        )
        if value is not None:
            print(list(shift.nums), shift.mark, value)

In [9]:
with bjason.Document(fn_unassigned, "r") as doc:
    mol = doc.mol_data[0]
    spec = mol.spectra[0]

    print(dir(spec))

IndexError: List index out of range